# Assignment 11: Production Defense-in-Depth Pipeline

**Student:** Võ Thanh Hiệp  
**Student ID:** 2A202600836  

This notebook runs the full Assignment 11 test suite:
- Rate limiter, input/output guardrails, LLM-as-Judge, audit log, monitoring
- Tests 1–4 from the assignment specification
- Bonus: session anomaly detector (6th safety layer)

## Setup

In [ ]:
import sys
from pathlib import Path

# Add src/ to path
SRC = Path("..").resolve() / "src"
sys.path.insert(0, str(SRC))

import os
from dotenv import load_dotenv
load_dotenv(Path("..").resolve() / ".env")

if not os.environ.get("GOOGLE_API_KEY"):
    from getpass import getpass
    os.environ["GOOGLE_API_KEY"] = getpass("GOOGLE_API_KEY: ")

print("Setup OK")

## Run All Assignment Tests

Executes Tests 1–4 plus output redaction and multi-criteria judge demos.

In [ ]:
import asyncio
from pipeline.defense_pipeline import run_all_assignment_tests

await run_all_assignment_tests(export_audit=True)

## Pipeline Components

The production pipeline is assembled in `src/pipeline/defense_pipeline.py`:

1. **AuditLogPlugin** — records every interaction
2. **RateLimitPlugin** — sliding window (10 req / 60s per user)
3. **SessionAnomalyPlugin** — bonus 6th layer for repeated injection attempts
4. **InputGuardrailPlugin** — regex injection + topic filter
5. **OutputGuardrailPlugin** — PII/secret redaction
6. **LlmJudgePlugin** — multi-criteria QA (safety, relevance, accuracy, tone)
7. **MonitoringAlert** — threshold alerts on block rates